In [1]:
# ==============================================================================
#  Шаг 0: Установка библиотек (если они еще не установлены)
# ==============================================================================
# Раскомментируйте и выполните следующую строку, если у вас нет этих библиотек
# !pip install numpy pandas plotly

# ==============================================================================
#  Шаг 1: Импорт библиотек и генерация данных
# ==============================================================================
import numpy as np
import pandas as pd
import plotly.express as px

print("Библиотеки успешно импортированы.")

# --- Генерация данных ---
# Создаем траекторию с положительными и отрицательными значениями
n_points = 100  # Количество точек во времени
n_dims = 5  # Размерность пространства
trajectory = np.zeros((n_points, n_dims))
half_point = n_points // 2

np.random.seed(42)  # для воспроизводимости

# Первая фаза траектории
trajectory[:half_point, 0] = (
    np.linspace(0.4, 0.5, half_point) + np.random.rand(half_point) * 0.1
)
trajectory[:half_point, 1] = (
    np.linspace(0.3, 0.4, half_point) + np.random.rand(half_point) * 0.1
)
trajectory[:half_point, 2] = (
    np.linspace(-0.2, -0.4, half_point) + np.random.rand(half_point) * 0.1
)
trajectory[half_point:, 0] = (
    np.linspace(-0.3, -0.4, half_point) + np.random.rand(half_point) * 0.1
)

# Вторая фаза траектории
trajectory[half_point:, 1] = (
    np.linspace(-0.2, -0.3, half_point) + np.random.rand(half_point) * 0.1
)
trajectory[half_point:, 2] = (
    np.linspace(0.4, 0.5, half_point) + np.random.rand(half_point) * 0.1
)
# Добавим более сложную динамику для оставшихся измерений
trajectory[:, 3] = np.sin(np.linspace(0, 4 * np.pi, n_points)) * 0.6
trajectory[:, 4] = np.cos(np.linspace(0, 2 * np.pi, n_points)) * 0.5 - np.linspace(
    0, 0.3, n_points
)

print(
    f"Сгенерированы данные: {trajectory.shape[0]} точек для {trajectory.shape[1]}-мерного вектора."
)

# ==============================================================================
#  Шаг 2: Преобразование данных и построение графика
# ==============================================================================

# --- Преобразование данных из широкого формата в длинный ---
# 1. Сначала создаем DataFrame в широком формате
column_names = [f"Измерение {i+1}" for i in range(n_dims)]
df_wide = pd.DataFrame(trajectory, columns=column_names)
df_wide["Время"] = np.arange(n_points)

# 2. Используем pd.melt для преобразования в длинный формат
df_long = df_wide.melt(
    id_vars=["Время"],  # Эта колонка останется как есть
    value_vars=column_names,  # Эти колонки будут "расплавлены"
    var_name="Измерение",  # Новая колонка для названий измерений
    value_name="Значение",  # Новая колонка для их значений
)

print("\nПример данных после преобразования (длинный формат):")
print(df_long.head())

# --- Построение интерактивного 3D графика ---
fig = px.scatter_3d(
    df_long,
    x="Время",
    y="Измерение",
    z="Значение",
    color="Измерение",  # Раскрашиваем точки по принадлежности к измерению
    symbol="Измерение",  # Используем разные символы (кружок, квадрат и т.д.)
    title="3D визуализация траектории в виде облака точек",
    labels={  # Задаем красивые подписи осей
        "Время": "Ось времени",
        "Измерение": "Компоненты вектора",
        "Значение": "Значение компонента",
    },
)

# --- Настройка внешнего вида сцены ---
fig.update_traces(marker=dict(size=4))  # Устанавливаем размер точек

fig.update_layout(
    width=900,
    height=700,
    scene=dict(
        # Устанавливаем начальное положение камеры для лучшего обзора
        camera=dict(eye=dict(x=1.8, y=-1.8, z=0.8))
    ),
    margin=dict(l=0, r=0, b=0, t=40),
)

# Показываем график
fig.show()

Библиотеки успешно импортированы.
Сгенерированы данные: 100 точек для 5-мерного вектора.

Пример данных после преобразования (длинный формат):
   Время    Измерение  Значение
0      0  Измерение 1  0.437454
1      1  Измерение 1  0.497112
2      2  Измерение 1  0.477281
3      3  Измерение 1  0.465988
4      4  Измерение 1  0.423765


In [2]:
# ==============================================================================
#  Шаг 0: Установка библиотек (если они еще не установлены)
# ==============================================================================
# Раскомментируйте и выполните следующую строку, если у вас нет этих библиотек
# !pip install numpy plotly

# ==============================================================================
#  Шаг 1: Импорт библиотек и генерация высокоразмерных данных
# ==============================================================================
import numpy as np
import plotly.graph_objects as go

print("Библиотеки успешно импортированы.")

# --- Генерация данных ---
# Используем те же самые данные для прямого сравнения методов
n_points = 100  # Количество точек во времени
n_dims = 1024  # Размерность пространства

print(f"Генерация {n_points} точек для {n_dims}-мерного вектора...")

# Создаем "волну" активности, которая движется по измерениям
trajectory = np.random.randn(n_points, n_dims) * 0.1  # Фоновый шум
center_of_wave = np.linspace(100, 900, n_points)
wave_width = 50
time_grid, dim_grid = np.meshgrid(np.arange(n_points), np.arange(n_dims), indexing="ij")
wave_activity = 2.0 * np.exp(
    -((dim_grid - center_of_wave[:, np.newaxis]) ** 2) / (2 * wave_width**2)
)
trajectory += wave_activity
trajectory[:, 500:520] += (
    np.sin(np.linspace(0, 4 * np.pi, n_points))[:, np.newaxis] * 1.5
)

print("Данные успешно сгенерированы.")


# ==============================================================================
#  Шаг 2: Построение 3D-поверхности
# ==============================================================================
print("\nНачинаю построение 3D-поверхности...")

# Создаем фигуру с помощью plotly.graph_objects
fig = go.Figure(
    data=[
        go.Surface(
            # Данные для поверхности. Мы транспонируем матрицу, чтобы
            # оси соответствовали нашему замыслу (Время по X, Измерения по Y).
            z=trajectory.T,
            # Задаем координаты для осей X и Y
            x=np.arange(n_points),
            y=np.arange(n_dims),
            # Выбираем цветовую шкалу. 'RdBu' (Red-Blue) отлично подходит
            # для данных, где есть положительные и отрицательные значения.
            colorscale="RdBu",
            # Фиксируем диапазон цвета, чтобы выбросы не "портили" шкалу
            cmin=-2.5,
            cmax=2.5,
            # Настраиваем шкалу цвета (colorbar)
            colorbar=dict(title="Значение"),
        )
    ]
)

# --- Настройка внешнего вида сцены и осей ---
fig.update_layout(
    title="1024-мерная траектория в виде 3D-поверхности",
    width=900,
    height=700,
    scene=dict(
        xaxis_title="Ось Времени",
        yaxis_title="Измерения (0-1023)",
        zaxis_title="Значение (визуализировано цветом и высотой)",
        # Устанавливаем начальное положение камеры для лучшего обзора
        camera_eye=dict(x=1.8, y=-1.8, z=1.5),
    ),
    margin=dict(l=0, r=0, b=0, t=50),
)

print("График построен. Идет отрисовка...")
fig.show()

Библиотеки успешно импортированы.
Генерация 100 точек для 1024-мерного вектора...
Данные успешно сгенерированы.

Начинаю построение 3D-поверхности...
График построен. Идет отрисовка...
